1. construct the DCOPF latex tutorial

2

In [3]:
%%time
import gurobipy as gp
from gurobipy import GRB
from matpowercaseframes import CaseFrames
import scipy.sparse as sp
import numpy as np
from collections import defaultdict
from numpy.linalg import inv

class Bus:
    def __init__(self, pd, gs, gens):
        self.pd = pd
        self.gs = gs
        self.gens = gens
class Gen:
    def __init__(self, bus, pmin, pmax, pstart, cost):
        self.bus = bus
        self.pmin = pmin
        self.pmax = pmax
        self.pstart = pstart
        self.cost = cost
class Line:
    def __init__(self, rate, frombus, tobus, one_over_reactance):
        self.rate = rate
        self.frombus = frombus
        self.tobus = tobus
        self.one_over_reactance = one_over_reactance # β beta
class NetworkReference:
    def __init__(self, ref, nbus, ngen, nline, r, bus, gen, line, originalindices, B, pi, stdw, line_prob=0.9, bus_prob=0.9):
        self.ref = ref # Dictionary of ref data
        self.nbus = nbus
        self.ngen = ngen
        self.nline = nline
        self.r = r  # Reference bus index
        self.bus = bus # list of bus
        self.gen = gen  
        self.line = line 
        self.originalindices = originalindices 
        self.B = B  # Admittance matrix
        self.pi = pi  # Inverse reduced admittance matrix
        self.stdw = stdw  # List of standard deviations
        self.line_prob = line_prob 
        self.bus_prob = bus_prob  
def admittancematrix(ref, bus_index):
    nbus = len(ref['bus'])
    B = np.zeros((nbus,nbus))
    bus_index_inv= {v:k for k,v in bus_index.items()}
    nline = len(ref['branch'])
    for br in range(nline):
        f_bus = bus_index_inv[ref['branch'].F_BUS.values[br]]
        t_bus = bus_index_inv[ref['branch'].T_BUS.values[br]]
        susceptance = ref['branch'].BR_X.values[br]/(ref['branch'].BR_X.values[br]**2+ref['branch'].BR_R.values[br]**2) # imaginary part of admittance, x/(x^2+r^2)
        B[f_bus-1, t_bus-1] += -susceptance
        B[t_bus-1, f_bus-1] += -susceptance
        B[f_bus-1, f_bus-1] += susceptance
        B[t_bus-1, t_bus-1] += susceptance
    return B, bus_index_inv
def cost(ref,p):
    return gp.quicksum(ref.gen[g].cost[0]*p[g] + ref.gen[g].cost[1]*p[g] + ref.gen[g].cost[2] for g in range(len(ref.gen)))
def NetworkReference_f(data_file, line_prob=0.9, bus_prob=0.9, sigma_scaling=0.05):
    mpc = CaseFrames(data_file)
    ref = {attr: getattr(mpc,attr) for attr in mpc.attributes}
    def generateindices(d):
        df = d.sort_values(by=d.columns[0]).reset_index(drop=True)
        originalindices = df.index+1
        nindices = len(originalindices)
        index_to_busID = dict(zip(originalindices,df[df.columns[0]])) # keys are unique, but busID at gen may not unique since there may more than one generation are connected to a single bus
        return nindices, originalindices, index_to_busID
    ngen, genindices, gen_index = generateindices(ref['gen'])
    bus_gens = defaultdict(list)
    for num, bus in enumerate(ref['bus'].BUS_I):
        if not bus in gen_index.values():
            bus_gens[bus] = []
        else:
            for num2, val in enumerate(gen_index.values()):
                if bus == val:
                    bus_gens[bus].append(list(gen_index.keys())[num2])
    gen = [Gen(
        ref['gen'].GEN_BUS.values[i],
        ref['gen'].PMIN.values[i],
        ref['gen'].PMAX.values[i],
        ref['gen'].PG.values[i],
        ref['gencost'][['COST_2','COST_1','COST_0']].values
        ) for i in range(ngen)]
    nbus, busindices, bus_index = generateindices(ref['bus'])
    bus = [Bus(
        ref['bus'].PD.values[i],
        ref['bus'].GS.values[i],
        bus_gens[bus_index[i+1]] ,
        ) for i in range(nbus)]
    nline, lineindices, fbus_index =generateindices(ref['branch'])
    line = [Line(
            ref['branch'].RATE_A.values[l],
            ref['branch'].F_BUS.values[l],
            ref['branch'].T_BUS.values[l],
            1/ ref['branch'].BR_X.values[l]
            ) for l in range(nline)]
    originalindices = {'bus':busindices, 'gen':genindices, 'line':lineindices}
    ref['ref_buses'] = ref['bus'][ref['bus'].BUS_TYPE==3]
    r = ref['ref_buses'].index[0]-1 # since the index of bus starting from 1
    nonref_indices = [b for b in range(nbus) if b != r]
    B, bus_index_inv = admittancematrix(ref, bus_index)
    pi = np.zeros((nbus,nbus))
    pi[np.ix_(nonref_indices,nonref_indices)] = inv(B[np.ix_(nonref_indices,nonref_indices)])
    stdw = [sigma_scaling*ref['bus'].PD.values[b] for b in range(nbus)]
    return NetworkReference(ref,nbus,ngen,nline,r,bus,gen,line,originalindices,B,pi,stdw,line_prob,bus_prob), bus_index_inv

data_file='julia\\pglib-opf-17.08\\pglib_opf_case300_ieee.m'
ref,bus_index_inv = NetworkReference_f(data_file, line_prob=0.9, bus_prob=0.9, sigma_scaling=0.05)

class SingleScenarioOPF:
    def __init__(self, model, p, omega):
        self.model = model
        self.p = p
        self.omega = omega
# def SingleScenarioOPF_f(ref):
model = gp.Model()
p = model.addMVar(shape=ref.ngen, vtype=GRB.CONTINUOUS, name='p',
                  lb=[ref.gen[g].pmin for g in range(ref.ngen)],
                  ub=[ref.gen[g].pmax for g in range(ref.ngen)])
p.start = [ref.gen[g].pstart for g in range(ref.ngen)]
omega = model.addMVar(shape=ref.nbus, vtype=GRB.CONTINUOUS, name='omega')
busvalue = {}
for i in range(ref.nbus):
    busvalue[i] = gp.quicksum(p[g] for g in range(len(ref.bus[i].gens))) + omega[i] - ref.bus[i].pd - ref.bus[i].gs
def theta(ref, busvalue, i):
    return gp.quicksum(ref.pi[i,j]*busvalue[j] for j in range(ref.nbus))
def lineflow(l):
    return ref.line[l].one_over_reactance*(
    theta(ref,busvalue,bus_index_inv[ref.line[l].frombus]-1) - theta(ref,busvalue,bus_index_inv[ref.line[l].tobus]-1)
    )
model.addConstrs((lineflow(l) <= ref.line[l].rate for l in range(ref.nline)), name='c1d_ub')

Warning for adding constraints: zero or small (< 1e-13) coefficients, ignored
CPU times: total: 50.4 s
Wall time: 51.7 s


{0: <MConstr ()>,
 1: <MConstr () *awaiting model update*>,
 2: <MConstr () *awaiting model update*>,
 3: <MConstr () *awaiting model update*>,
 4: <MConstr () *awaiting model update*>,
 5: <MConstr () *awaiting model update*>,
 6: <MConstr () *awaiting model update*>,
 7: <MConstr () *awaiting model update*>,
 8: <MConstr () *awaiting model update*>,
 9: <MConstr () *awaiting model update*>,
 10: <MConstr () *awaiting model update*>,
 11: <MConstr () *awaiting model update*>,
 12: <MConstr () *awaiting model update*>,
 13: <MConstr () *awaiting model update*>,
 14: <MConstr () *awaiting model update*>,
 15: <MConstr () *awaiting model update*>,
 16: <MConstr () *awaiting model update*>,
 17: <MConstr () *awaiting model update*>,
 18: <MConstr () *awaiting model update*>,
 19: <MConstr () *awaiting model update*>,
 20: <MConstr () *awaiting model update*>,
 21: <MConstr () *awaiting model update*>,
 22: <MConstr () *awaiting model update*>,
 23: <MConstr () *awaiting model update*>,
 2